# Load one quarter of `changed_holdings` into local Neo4j

Bipartite graph for `(YEAR, QUARTER)`:
- `(:Investor {cik, aum, year, quarter})`
- `(:Stock {cusip, log_return, true_class, year, quarter})`
- `(:Investor)-[:CHANGED {change_in_weight, change_in_adjusted_weight, change_in_shares}]->(:Stock)`

**Pre-req**: Neo4j Desktop (or Docker) running locally on `bolt://localhost:7687`. Add `NEO4J_PASSWORD=<your-password>` to `.env`.

In [ ]:
# Run once if the driver isn't installed
# %pip install neo4j

In [2]:
import os
import sys
from getpass import getpass
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from neo4j import GraphDatabase

for env_path in [Path.cwd() / ".env", Path.cwd().parent / ".env", Path.cwd().parent.parent / ".env"]:
    if env_path.exists():
        load_dotenv(env_path)
        print("Loaded .env from:", env_path.resolve())
        break

def project_root() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, cwd.parent, cwd.parent.parent]:
        if (p / "ETL").is_dir():
            return p
    return cwd.parent.parent

ROOT = project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ETL.gnn_db_pipeline.config import (
    TARGET_DB,
    TGT_CHANGED_HOLDINGS,
    TGT_CHANGED_STAS,
    TGT_CIK_AUM,
    TGT_STOCKS_RETURN,
)
from ETL.gnn_db_pipeline.db_connector import ConfigurablePostgresHandler

pg = ConfigurablePostgresHandler(TARGET_DB)
pg.connect()
print("Connected to Postgres:", TARGET_DB)

2026-04-18 22:07:26 - ETL_Pipeline - INFO - Connected to PostgreSQL: postgres@127.0.0.1:5432/13FGNN


Loaded .env from: C:\Users\potda\Daniel\BGU\Year_D\סמסטר ז\Final_Project\Social-Network-Stock-Market\.env
Failed to delete log file logs\etl_pipeline_20260418_215754.log: [WinError 32] The process cannot access the file because it is being used by another process: 'logs\\etl_pipeline_20260418_215754.log'
Failed to delete log file logs\etl_pipeline_20260418_191736.log: [WinError 32] The process cannot access the file because it is being used by another process: 'logs\\etl_pipeline_20260418_191736.log'
Connected to Postgres: 13FGNN


## Config

In [3]:
YEAR, QUARTER = 2022, 2

NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "test1234"
NEO4J_DATABASE = "neo4j"

WIPE_FIRST = True       # delete all nodes/edges in the DB before loading
BATCH_SIZE = 5000       # edges per UNWIND batch

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print("Connected to Neo4j at", NEO4J_URI)

Connected to Neo4j at bolt://localhost:7687


## Pull data from Postgres

In [4]:
edges = pg.query(f"""
    SELECT cik, cusip, change_in_shares, change_in_weight, change_in_adjusted_weight
    FROM {TGT_CHANGED_HOLDINGS}
    WHERE year = %s AND quarter = %s
""", (YEAR, QUARTER))

aum = pg.query(f"""
    SELECT cik, total AS aum
    FROM {TGT_CIK_AUM}
    WHERE year = %s AND quarter = %s
""", (YEAR, QUARTER))

returns = pg.query(f"""
    SELECT cusip, log_return
    FROM {TGT_STOCKS_RETURN}
    WHERE year = %s AND quarter = %s
""", (YEAR, QUARTER))

tertiles = pg.query(f"""
    SELECT log_return_tertile_1, log_return_tertile_2
    FROM {TGT_CHANGED_STAS}
    WHERE year = %s AND quarter = %s
""", (YEAR, QUARTER))
t1 = float(tertiles.iloc[0, 0])
t2 = float(tertiles.iloc[0, 1])

returns["true_class"] = np.where(
    returns["log_return"] <= t1, 0,
    np.where(returns["log_return"] <= t2, 1, 2),
)

print(f"Edges: {len(edges):,}")
print(f"Investors w/ AUM: {len(aum):,}")
print(f"Stocks w/ return: {len(returns):,}")
print(f"Tertile cutoffs: t1={t1:.4f}  t2={t2:.4f}")

C:\Users\potda\Daniel\BGU\Year_D\סמסטר ז\Final_Project\Social-Network-Stock-Market\ETL\data_handlers\db_data_handler\postgres_handler.py:456: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, self.connection, params=params)


Edges: 1,104,392
Investors w/ AUM: 6,537
Stocks w/ return: 3,433
Tertile cutoffs: t1=-0.2768  t2=-0.1028


In [5]:
# Nodes = distinct participants in the quarter (some may have no AUM / no return)
investors = (
    pd.DataFrame({"cik": edges["cik"].unique()})
    .merge(aum, on="cik", how="left")
)
investors["year"] = YEAR
investors["quarter"] = QUARTER

stocks = (
    pd.DataFrame({"cusip": edges["cusip"].unique()})
    .merge(returns, on="cusip", how="left")
)
stocks["year"] = YEAR
stocks["quarter"] = QUARTER

print("Investor rows:", len(investors), " | missing AUM:", investors["aum"].isna().sum())
print("Stock rows:   ", len(stocks),    " | missing return:", stocks["log_return"].isna().sum())

Investor rows: 6428  | missing AUM: 0
Stock rows:    3149  | missing return: 35


## Neo4j schema + optional wipe

In [6]:
def run_write(cypher: str, params: dict | None = None):
    with driver.session(database=NEO4J_DATABASE) as s:
        return s.execute_write(lambda tx: tx.run(cypher, **(params or {})).consume())

def run_read(cypher: str, params: dict | None = None):
    with driver.session(database=NEO4J_DATABASE) as s:
        return s.execute_read(lambda tx: list(tx.run(cypher, **(params or {}))))

if WIPE_FIRST:
    # Detach-delete in batches to avoid blowing memory on large graphs
    run_write("CALL apoc.periodic.iterate('MATCH (n) RETURN n', 'DETACH DELETE n', {batchSize: 10000}) YIELD batches, total RETURN batches, total")\
        if False else run_write("MATCH (n) DETACH DELETE n")
    print("Database wiped.")

# Uniqueness constraints (also creates the backing index)
run_write("CREATE CONSTRAINT investor_cik_unique IF NOT EXISTS FOR (i:Investor) REQUIRE i.cik IS UNIQUE")
run_write("CREATE CONSTRAINT stock_cusip_unique   IF NOT EXISTS FOR (s:Stock)    REQUIRE s.cusip IS UNIQUE")
print("Constraints ensured.")

Database wiped.
Constraints ensured.


## Write nodes + edges (UNWIND batches)

In [7]:
def df_to_records(df: pd.DataFrame) -> list[dict]:
    # Convert NaN -> None so Neo4j stores missing props as null.
    return df.astype(object).where(pd.notna(df), None).to_dict(orient="records")


def batched(records: list[dict], size: int):
    for i in range(0, len(records), size):
        yield records[i : i + size]


CYPHER_INVESTORS = """
UNWIND $rows AS row
MERGE (i:Investor {cik: row.cik})
SET i.aum = row.aum,
    i.year = row.year,
    i.quarter = row.quarter
"""

CYPHER_STOCKS = """
UNWIND $rows AS row
MERGE (s:Stock {cusip: row.cusip})
SET s.log_return = row.log_return,
    s.true_class = row.true_class,
    s.year = row.year,
    s.quarter = row.quarter
"""

CYPHER_EDGES = """
UNWIND $rows AS row
MATCH (i:Investor {cik: row.cik})
MATCH (s:Stock {cusip: row.cusip})
MERGE (i)-[r:CHANGED]->(s)
SET r.change_in_weight = row.change_in_weight,
    r.change_in_adjusted_weight = row.change_in_adjusted_weight,
    r.change_in_shares = row.change_in_shares,
    r.year = $year,
    r.quarter = $quarter
"""

inv_records = df_to_records(investors)
stk_records = df_to_records(stocks)
edge_records = df_to_records(edges)

run_write(CYPHER_INVESTORS, {"rows": inv_records})
print(f"Wrote {len(inv_records):,} investors.")

run_write(CYPHER_STOCKS, {"rows": stk_records})
print(f"Wrote {len(stk_records):,} stocks.")

total_edges = 0
for chunk in batched(edge_records, BATCH_SIZE):
    run_write(CYPHER_EDGES, {"rows": chunk, "year": YEAR, "quarter": QUARTER})
    total_edges += len(chunk)
print(f"Wrote {total_edges:,} edges in batches of {BATCH_SIZE}.")

Wrote 6,428 investors.
Wrote 3,149 stocks.
Wrote 1,104,392 edges in batches of 5000.


## Verify

In [8]:
counts = run_read("""
MATCH (i:Investor) WITH count(i) AS n_investors
MATCH (s:Stock)    WITH n_investors, count(s) AS n_stocks
MATCH ()-[r:CHANGED]->() RETURN n_investors, n_stocks, count(r) AS n_edges
""")[0]
print("Neo4j counts:", dict(counts))

sample = run_read("""
MATCH (i:Investor)-[r:CHANGED]->(s:Stock)
RETURN i.cik AS cik, i.aum AS aum, s.cusip AS cusip,
       s.log_return AS log_return, s.true_class AS true_class,
       r.change_in_weight AS w, r.change_in_adjusted_weight AS w_adj
ORDER BY r.change_in_weight DESC
LIMIT 10
""")
pd.DataFrame([dict(r) for r in sample])

Neo4j counts: {'n_investors': 6428, 'n_stocks': 3149, 'n_edges': 1104392}


,cik,aum,cusip,log_return,true_class,w,w_adj
0,0001803460,1.698339e+05,22266M104,-0.485399,0.0,1.0,0.111273
1,0002034001,1.054000e+06,83406F102,-0.583984,0.0,1.0,1.602566
2,0001801811,4.295700e+05,00437E102,-0.864144,0.0,1.0,0.002102
3,0001279926,8.645901e+04,345370860,-0.411627,0.0,1.0,2.640871
4,0001534949,2.046024e+06,683712103,-0.607871,0.0,1.0,1.268066
5,0001632833,2.876947e+07,002896207,-0.636927,0.0,1.0,1.549937
6,0001279396,2.336535e+06,207410101,-0.437002,0.0,1.0,0.232491
7,0001569356,2.765495e+06,98850P109,0.157700,2.0,1.0,0.956351
8,0001616168,1.008995e+06,023135106,-0.428317,0.0,1.0,0.018412
9,0001688150,7.643300e+05,63910B102,-0.856237,0.0,1.0,0.016049


## Example queries to try in Neo4j Browser

```cypher
// Top 10 stocks by positive-Δ degree (how many investors increased exposure)
MATCH (i:Investor)-[r:CHANGED]->(s:Stock)
WHERE r.change_in_weight > 0
RETURN s.cusip, s.true_class, count(*) AS n_buyers
ORDER BY n_buyers DESC LIMIT 10;

// Biggest investors by AUM and how many stocks they touched
MATCH (i:Investor)-[:CHANGED]->(s:Stock)
RETURN i.cik, i.aum, count(DISTINCT s) AS n_stocks
ORDER BY i.aum DESC LIMIT 10;

// High-tertile stocks and their biggest buyers
MATCH (i:Investor)-[r:CHANGED]->(s:Stock {true_class: 2})
RETURN s.cusip, i.cik, r.change_in_weight
ORDER BY r.change_in_weight DESC LIMIT 20;
```

In [ ]:
driver.close()
pg.disconnect() if hasattr(pg, "disconnect") else None
print("Connections closed.")